# MDD Challenge 2025 - CTC + PER-MDD Public Test Inference

This notebook keeps the original train/validation split from `audio-per-mdd.ipynb`, tunes the PER-MDD retrieval layer on the validation split, then runs inference on the public test set and writes `predictions.csv` with columns `id,predict`.

Important constraints:
- The public test `transcript` column is **not** used for training, tuning, or inference.
- Test row order is defined as lexicographic filename order inside `audio_data/public_test`.
- Output `id` starts at `0` and follows that folder order exactly.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import re
import csv
import wave
import math
import random
import inspect
import subprocess
import sys
import json
import importlib.util
from collections import Counter, defaultdict
from dataclasses import dataclass

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f'Installing missing package: {package}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

ensure_package('torch')
ensure_package('sklearn', 'scikit-learn')
ensure_package('transformers')
ensure_package('accelerate')

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoFeatureExtractor,
    Trainer,
    TrainingArguments,
    Wav2Vec2ForCTC,
    get_linear_schedule_with_warmup,
    set_seed,
)

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 100)

BASELINE_SEED = 42
set_seed(BASELINE_SEED)
random.seed(BASELINE_SEED)
np.random.seed(BASELINE_SEED)
torch.manual_seed(BASELINE_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Load Training Data


In [ ]:
def find_dataset_root():
    candidates = [
        Path('/kaggle/input/mdd-challenge-2025-training-set/MDD-Challenge-2025-training-set'),
        Path('/kaggle/input/mdd-challenge-2025-training-set'),
        Path('/kaggle/input/mdd-challenge-data/MDD-Challenge-2025-training-set'),
        Path('/kaggle/input/mdd-challenge-data'),
        Path('./MDD-Challenge-2025-training-set'),
        Path('.'),
    ]
    for p in candidates:
        if (p / 'metadata' / 'train.csv').exists() and (p / 'metadata' / 'train_phones.csv').exists():
            return p
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for train_csv in kaggle_input.rglob('metadata/train.csv'):
            root = train_csv.parent.parent
            if (root / 'metadata' / 'train_phones.csv').exists():
                return root
    for train_csv in Path('.').rglob('metadata/train.csv'):
        root = train_csv.parent.parent
        if (root / 'metadata' / 'train_phones.csv').exists():
            return root
    raise FileNotFoundError('Training dataset root not found. Please set DATA_ROOT manually.')

DATA_ROOT = find_dataset_root()
META_DIR = DATA_ROOT / 'metadata'
AUDIO_DIR = DATA_ROOT / 'audio_data' / 'train'
TRAIN_CSV = META_DIR / 'train.csv'
TRAIN_PHONES_CSV = META_DIR / 'train_phones.csv'

train_text = pd.read_csv(TRAIN_CSV)
train_phones = pd.read_csv(TRAIN_PHONES_CSV)

def normalize_phone_text(s):
    return ' '.join(str(s).replace('*', '').replace('$', '').split())

def eval_phone_tokens(s):
    return normalize_phone_text(s).split()

def resolve_audio_path(rel_path):
    rel = Path(str(rel_path))
    candidates = [DATA_ROOT / rel, DATA_ROOT / 'audio_data' / 'train' / rel.name, AUDIO_DIR / rel.name]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

def infer_speaker_group(sample_id, rel_path):
    stem = Path(str(rel_path)).stem
    has_long_timestamp_suffix = bool(re.search(r'[_-]\d{10,}$', stem)) or bool(re.search(r'[_-]\d{10,}$', str(sample_id)))
    return 'adult' if has_long_timestamp_suffix else 'child'

train_phones['canonical_norm'] = train_phones['canonical'].map(normalize_phone_text)
train_phones['transcript_norm'] = train_phones['transcript'].map(normalize_phone_text)
train_phones['audio_path'] = train_phones['path'].map(resolve_audio_path)
train_phones['audio_exists'] = train_phones['audio_path'].map(lambda p: p.exists())
train_phones['speaker_group'] = [infer_speaker_group(i, p) for i, p in zip(train_phones['id'], train_phones['path'])]
train_phones['has_pron_error'] = train_phones['canonical_norm'] != train_phones['transcript_norm']

print('DATA_ROOT:', DATA_ROOT)
print('Rows:', len(train_phones))
print('Missing audio:', int((~train_phones['audio_exists']).sum()))
display(train_phones[['id', 'path', 'speaker_group', 'canonical_norm', 'transcript_norm']].head())


## 3. Alignment And Metrics


In [ ]:
def align(seq1, seq2):
    GAP = -1
    MATCH = 1
    MISMATCH = -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        score[i][0] = GAP * i
    for j in range(n + 1):
        score[0][j] = GAP * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[j - 1] == seq2[i - 1]:
                s = MATCH
            elif seq1[j - 1] == '<eps>' or seq2[i - 1] == '<eps>':
                s = GAP
            else:
                s = MISMATCH
            score[i][j] = max(score[i - 1][j - 1] + s, score[i - 1][j] + GAP, score[i][j - 1] + GAP)
    align1, align2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        if seq1[j - 1] == seq2[i - 1]:
            s = MATCH
        elif seq1[j - 1] == '<eps>' or seq2[i - 1] == '<eps>':
            s = GAP
        else:
            s = MISMATCH
        if score[i][j] == score[i - 1][j - 1] + s:
            align1.append(seq1[j - 1]); align2.append(seq2[i - 1])
            i -= 1; j -= 1
        elif score[i][j] == score[i][j - 1] + GAP:
            align1.append(seq1[j - 1]); align2.append('<eps>')
            j -= 1
        else:
            align1.append('<eps>'); align2.append(seq2[i - 1])
            i -= 1
    while j > 0:
        align1.append(seq1[j - 1]); align2.append('<eps>'); j -= 1
    while i > 0:
        align1.append('<eps>'); align2.append(seq2[i - 1]); i -= 1
    align1.reverse(); align2.reverse()
    return align1, align2

def ops(aligned1, aligned2):
    out = []
    for r, h in zip(aligned1, aligned2):
        if r != '<eps>' and h == '<eps>':
            out.append('D')
        elif r == '<eps>' and h != '<eps>':
            out.append('I')
        elif r != h:
            out.append('S')
        else:
            out.append('C')
    return out

def align_pair(s1, s2):
    a1, a2 = align(eval_phone_tokens(s1), eval_phone_tokens(s2))
    return a1, a2, ops(a1, a2)

def transcript_correct_mask(canonical, transcript):
    ref_seq, hyp_seq, op = align_pair(canonical, transcript)
    mask = []
    for ref, hyp, o in zip(ref_seq, hyp_seq, op):
        if hyp != '<eps>':
            mask.append(o == 'C')
    return mask

def canonical_to_raw_map(canonical, raw_predict):
    ref_seq, raw_seq, _ = align_pair(canonical, raw_predict)
    mapped = []
    for ref, raw in zip(ref_seq, raw_seq):
        if ref != '<eps>':
            mapped.append(None if raw == '<eps>' else raw)
    return mapped

def compute_metrics_from_frames(ground_truth_df, results_df):
    cor_cor = cor_nocor = 0
    sub_sub = sub_sub1 = sub_nosub = 0
    ins_ins = ins_ins1 = ins_noins = 0
    del_del = del_del1 = del_nodel = 0
    total_sub = total_del = total_ins = total_ref_len = 0

    for gt_row, res_row in zip(ground_truth_df.to_dict('records'), results_df.to_dict('records')):
        ref_str = gt_row['canonical']
        human_str = gt_row['transcript']
        our_str = res_row['predict']

        ref_seq, human_seq, op_rh = align_pair(ref_str, human_str)
        human_seq2, our_seq2, op_ho = align_pair(human_str, our_str)
        ref_seq3, our_seq3, op_ro = align_pair(ref_str, our_str)

        total_sub += op_ho.count('S')
        total_del += op_ho.count('D')
        total_ins += op_ho.count('I')
        total_ref_len += op_ho.count('S') + op_ho.count('D') + op_ho.count('C')

        flag = 0
        for i in range(len(ref_seq)):
            if ref_seq[i] == '<eps>':
                continue
            while flag < len(ref_seq3) and ref_seq3[flag] == '<eps>':
                flag += 1
            if flag < len(ref_seq3) and ref_seq[i] == ref_seq3[flag]:
                if op_rh[i] == 'D' and op_ro[flag] == 'D':
                    del_del += 1
                elif op_rh[i] == 'D' and op_ro[flag] != 'D' and op_ro[flag] != 'C':
                    del_del1 += 1
                elif op_rh[i] == 'D' and op_ro[flag] != 'D' and op_ro[flag] == 'C':
                    del_nodel += 1
                flag += 1

        flag = 0
        for i in range(len(human_seq)):
            if human_seq[i] == '<eps>':
                continue
            while flag < len(human_seq2) and human_seq2[flag] == '<eps>':
                flag += 1
            if flag < len(human_seq2) and human_seq[i] == human_seq2[flag]:
                if op_rh[i] == 'C' and op_ho[flag] == 'C':
                    cor_cor += 1
                elif op_rh[i] == 'C' and op_ho[flag] != 'C':
                    cor_nocor += 1
                if op_rh[i] == 'S' and op_ho[flag] == 'C':
                    sub_sub += 1
                elif op_rh[i] == 'S' and op_ho[flag] != 'C' and ref_seq[i] != our_seq2[flag]:
                    sub_sub1 += 1
                elif op_rh[i] == 'S' and op_ho[flag] != 'C' and ref_seq[i] == our_seq2[flag]:
                    sub_nosub += 1
                if op_rh[i] == 'I' and op_ho[flag] == 'C':
                    ins_ins += 1
                elif op_rh[i] == 'I' and op_ho[flag] != 'C' and op_ho[flag] != 'D':
                    ins_ins1 += 1
                elif op_rh[i] == 'I' and op_ho[flag] != 'C' and op_ho[flag] == 'D':
                    ins_noins += 1
                flag += 1

    TR = sub_sub + sub_sub1 + del_del + del_del1 + ins_ins + ins_ins1
    FR = cor_nocor
    FA = sub_nosub + ins_noins + del_nodel
    DE = sub_sub1 + del_del1 + ins_ins1
    precision = TR / (TR + FR) if (TR + FR) > 0 else 0.0
    recall = TR / (TR + FA) if (TR + FA) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    per = (total_sub + total_del + total_ins) / total_ref_len if total_ref_len > 0 else 0.0
    der = DE / (TR + FA) if (TR + FA) > 0 else 0.0
    score = 0.5 * f1 + 0.4 * (1 - der) + 0.1 * (1 - per)
    return {
        'F1': f1,
        'Precision': precision,
        'Recall': recall,
        'PER': per,
        'DER': der,
        'Score': score,
        'TR': TR,
        'FR': FR,
        'FA': FA,
        'DE': DE,
    }

def print_metrics(name, metrics):
    print(name)
    for key in ['F1', 'Precision', 'Recall', 'PER', 'DER', 'Score']:
        print(f'  {key}: {metrics[key]:.6f}')
    print(f"  Counts: TR={metrics['TR']} FR={metrics['FR']} FA={metrics['FA']} DE={metrics['DE']}")


## 4. Vocabulary And Config


In [ ]:
transcript_phone_vocab = Counter(tok for s in train_phones['transcript_norm'] for tok in eval_phone_tokens(s))
phone_tokens_sorted = sorted(transcript_phone_vocab.keys())
blank_token = '<blank>'
unk_token = '<unk>'
id2phone = [blank_token, unk_token] + phone_tokens_sorted
phone2id = {p: i for i, p in enumerate(id2phone)}

def encode_phone_sequence(s):
    return [phone2id.get(tok, phone2id[unk_token]) for tok in eval_phone_tokens(s)]

def decode_phone_ids(ids, collapse_ctc=False):
    out = []
    prev = None
    for idx in ids:
        idx = int(idx)
        if collapse_ctc and idx == prev:
            prev = idx
            continue
        prev = idx
        if idx == phone2id[blank_token]:
            continue
        out.append(id2phone[idx] if 0 <= idx < len(id2phone) else unk_token)
    return ' '.join(out)

print('CTC vocab size:', len(id2phone))
print('blank id:', phone2id[blank_token], 'unk id:', phone2id[unk_token])

MODEL_NAME = 'nguyenvulebinh/wav2vec2-base-vietnamese-250h'
SAMPLING_RATE = 16000

# T4-safe final config for challenge runs.
FAST_DEV_RUN = False
USE_SUBSET = False
MAX_TRAIN_SAMPLES = 800
MAX_VALID_SAMPLES = 200
NUM_EPOCHS = 20

PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
ENCODER_LEARNING_RATE = 2e-5
HEAD_LEARNING_RATE = 2e-3
LEARNING_RATE = ENCODER_LEARNING_RATE
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
FP16 = torch.cuda.is_available()
FREEZE_FEATURE_EXTRACTOR = True
DISABLE_MODEL_SPEC_AUGMENT = True

BLANK_DECODE_PENALTY = 0.5
UNK_DECODE_PENALTY = 10.0

POOL_FRAME_STRATEGY = 'middle'
POOL_MAX_PER_PHONE = 2000
INFER_BATCH_SIZE = 2

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
BASELINE_RUN_DIR = OUT_DIR / 'phoneme_ctc_baseline_public_test'
PER_MDD_DIR = OUT_DIR / 'phoneme_ctc_per_mdd_public_test'
BASELINE_RUN_DIR.mkdir(parents=True, exist_ok=True)
PER_MDD_DIR.mkdir(parents=True, exist_ok=True)

print('MODEL_NAME:', MODEL_NAME)
print('RUN_PROFILE: T4-safe final')
print('FAST_DEV_RUN:', FAST_DEV_RUN)
print('USE_SUBSET:', USE_SUBSET)
print('NUM_EPOCHS:', NUM_EPOCHS)
print('PER_DEVICE_BATCH_SIZE:', PER_DEVICE_BATCH_SIZE)
print('GRAD_ACCUM_STEPS:', GRAD_ACCUM_STEPS)
print('BASELINE_RUN_DIR:', BASELINE_RUN_DIR)
print('PER_MDD_DIR:', PER_MDD_DIR)


## 5. Train/Validation Split


In [ ]:
baseline_df = train_phones.copy().reset_index(drop=True)
baseline_df['audio_path_str'] = baseline_df['audio_path'].map(str)
baseline_df['label_text'] = baseline_df['transcript_norm']
baseline_df['stratify_key'] = baseline_df['speaker_group'].astype(str) + '_' + baseline_df['has_pron_error'].astype(int).astype(str)

stratum_counts = baseline_df['stratify_key'].value_counts()
rare_strata = set(stratum_counts[stratum_counts < 2].index)
baseline_df.loc[baseline_df['stratify_key'].isin(rare_strata), 'stratify_key'] = baseline_df.loc[
    baseline_df['stratify_key'].isin(rare_strata), 'speaker_group'
].astype(str)

train_idx, valid_idx = train_test_split(
    np.arange(len(baseline_df)),
    test_size=0.15,
    random_state=BASELINE_SEED,
    stratify=baseline_df['stratify_key'],
)
train_df = baseline_df.iloc[train_idx].reset_index(drop=True)
valid_df = baseline_df.iloc[valid_idx].reset_index(drop=True)

if USE_SUBSET:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=BASELINE_SEED).reset_index(drop=True)
    valid_df = valid_df.sample(min(MAX_VALID_SAMPLES, len(valid_df)), random_state=BASELINE_SEED).reset_index(drop=True)

split_dir = PER_MDD_DIR / 'splits'
split_dir.mkdir(parents=True, exist_ok=True)
train_df[['id', 'path', 'speaker_group', 'has_pron_error']].to_csv(split_dir / 'train_split_ids.csv', index=False)
valid_df[['id', 'path', 'speaker_group', 'has_pron_error']].to_csv(split_dir / 'valid_split_ids.csv', index=False)

print('train rows:', len(train_df))
print('valid rows:', len(valid_df))
display(train_df['has_pron_error'].value_counts(dropna=False).rename_axis('has_pron_error').reset_index(name='train_count'))
display(valid_df['has_pron_error'].value_counts(dropna=False).rename_axis('has_pron_error').reset_index(name='valid_count'))


## 6. Dataset, Model, And Baseline Training


In [ ]:
def read_wav_pcm(path):
    with wave.open(str(path), 'rb') as wf:
        sr = wf.getframerate()
        n_channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        frames = wf.readframes(wf.getnframes())
    if sampwidth == 2:
        data = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
    elif sampwidth == 4:
        data = np.frombuffer(frames, dtype=np.int32).astype(np.float32) / 2147483648.0
    elif sampwidth == 1:
        data = (np.frombuffer(frames, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    else:
        raise ValueError(f'Unsupported sample width: {sampwidth}')
    if n_channels > 1:
        data = data.reshape(-1, n_channels).mean(axis=1)
    return data.astype(np.float32), sr

class PhonemeCtcDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y, sr = read_wav_pcm(row['audio_path'])
        if sr != SAMPLING_RATE:
            raise ValueError(f'Expected {SAMPLING_RATE} Hz, got {sr}: {row["audio_path"]}')
        return {
            'input_values': y,
            'labels': encode_phone_sequence(row['label_text']),
            'id': row['id'],
        }

@dataclass
class CtcDataCollator:
    feature_extractor: object
    sampling_rate: int = SAMPLING_RATE
    label_pad_id: int = -100

    def __call__(self, features):
        input_values = [f['input_values'] for f in features]
        batch = self.feature_extractor(
            input_values,
            sampling_rate=self.sampling_rate,
            padding=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        labels = [torch.tensor(f['labels'], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in labels)
        padded = torch.full((len(labels), max_len), self.label_pad_id, dtype=torch.long)
        for i, label in enumerate(labels):
            padded[i, : len(label)] = label
        batch['labels'] = padded
        return batch

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
train_dataset = PhonemeCtcDataset(train_df)
valid_dataset = PhonemeCtcDataset(valid_df)
data_collator = CtcDataCollator(feature_extractor=feature_extractor)

sample_item = train_dataset[0]
print('sample audio shape:', sample_item['input_values'].shape)
print('sample label len:', len(sample_item['labels']))
print('sample decoded label:', decode_phone_ids(sample_item['labels']))

def load_or_init_model():
    final_model_dir = BASELINE_RUN_DIR / 'final_model'
    source = final_model_dir if (final_model_dir / 'config.json').exists() else MODEL_NAME
    print('Loading model from:', source)
    model = Wav2Vec2ForCTC.from_pretrained(
        source,
        vocab_size=len(id2phone),
        pad_token_id=phone2id[blank_token],
        ctc_loss_reduction='mean',
        ctc_zero_infinity=True,
        ignore_mismatched_sizes=True,
    )
    model.config.pad_token_id = phone2id[blank_token]
    model.config.ctc_zero_infinity = True
    model.config.ctc_loss_reduction = 'mean'
    if DISABLE_MODEL_SPEC_AUGMENT:
        model.config.apply_spec_augment = False
        model.config.mask_time_prob = 0.0
    if FREEZE_FEATURE_EXTRACTOR:
        if hasattr(model, 'freeze_feature_encoder'):
            model.freeze_feature_encoder()
        else:
            model.freeze_feature_extractor()
    return model

model = load_or_init_model().to(DEVICE)

with torch.no_grad():
    sample_batch = data_collator([train_dataset[i] for i in range(min(2, len(train_dataset)))])
    sample_batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in sample_batch.items()}
    sample_loss = model(**sample_batch).loss

print('vocab_size:', model.config.vocab_size)
print('blank/pad id:', model.config.pad_token_id)
print('lm_head:', model.lm_head)
print('initial sample CTC loss:', float(sample_loss.detach().cpu()))

final_model_dir = BASELINE_RUN_DIR / 'final_model'
if not (final_model_dir / 'config.json').exists():
    print('No baseline final_model found. Training baseline CTC model...')
    training_kwargs = dict(
        output_dir=str(BASELINE_RUN_DIR / 'checkpoints'),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        fp16=FP16,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=1,
        report_to='none',
        remove_unused_columns=False,
        dataloader_num_workers=0,
        group_by_length=False,
    )
    training_arg_params = inspect.signature(TrainingArguments.__init__).parameters
    if 'eval_strategy' in training_arg_params:
        training_kwargs['eval_strategy'] = 'epoch'
    else:
        training_kwargs['evaluation_strategy'] = 'epoch'
    training_args = TrainingArguments(**training_kwargs)

    head_param_ids = {id(p) for p in model.lm_head.parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_param_ids and p.requires_grad]
    encoder_params = [p for p in model.parameters() if id(p) not in head_param_ids and p.requires_grad]
    optimizer = torch.optim.AdamW(
        [
            {'params': encoder_params, 'lr': ENCODER_LEARNING_RATE},
            {'params': head_params, 'lr': HEAD_LEARNING_RATE},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    steps_per_epoch = math.ceil(len(train_dataset) / (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
    num_training_steps = max(1, int(steps_per_epoch * NUM_EPOCHS))
    num_warmup_steps = int(num_training_steps * WARMUP_RATIO)
    lr_scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )
    print('encoder lr:', ENCODER_LEARNING_RATE, 'head lr:', HEAD_LEARNING_RATE)
    print('training steps:', num_training_steps, 'warmup steps:', num_warmup_steps)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        data_collator=data_collator,
        optimizers=(optimizer, lr_scheduler),
    )
    train_result = trainer.train()
    trainer.save_model(str(final_model_dir))
    feature_extractor.save_pretrained(str(final_model_dir))
    print(train_result)
else:
    print('Using existing baseline final_model:', final_model_dir)


## 7. Inference With Cropped Decode And Hidden States


In [ ]:
def get_feature_lengths_from_attention(attention_mask):
    input_lengths = attention_mask.sum(-1)
    return model._get_feat_extract_output_lengths(input_lengths).to(torch.long)

def greedy_decode_one(logits, feature_length=None, blank_penalty=0.0, unk_penalty=0.0):
    adjusted = logits.copy()
    if feature_length is not None:
        adjusted = adjusted[: int(feature_length)]
    if blank_penalty:
        adjusted[:, phone2id[blank_token]] -= float(blank_penalty)
    if unk_penalty:
        adjusted[:, phone2id[unk_token]] -= float(unk_penalty)
    pred_ids = np.argmax(adjusted, axis=-1)
    return decode_phone_ids(pred_ids.tolist(), collapse_ctc=True)

def run_manual_inference(df, batch_size=INFER_BATCH_SIZE):
    model.eval()
    records = []
    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start + batch_size]
        waves = []
        for p in batch_df['audio_path']:
            y, sr = read_wav_pcm(p)
            if sr != SAMPLING_RATE:
                raise ValueError(f'Expected {SAMPLING_RATE} Hz, got {sr}: {p}')
            waves.append(y)
        inputs = feature_extractor(
            waves,
            sampling_rate=SAMPLING_RATE,
            padding=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
        logits = outputs.logits.detach().cpu().numpy()
        hidden = outputs.hidden_states[-1].detach().cpu().numpy()
        feat_lens = get_feature_lengths_from_attention(inputs['attention_mask']).detach().cpu().numpy()
        for local_i, (_, row) in enumerate(batch_df.iterrows()):
            T = int(feat_lens[local_i])
            records.append({
                'id': row['id'],
                'path': row.get('path', None),
                'canonical_norm': row.get('canonical_norm', None),
                'transcript_norm': row.get('transcript_norm', None),
                'logits': logits[local_i, :T].astype(np.float32),
                'hidden': hidden[local_i, :T].astype(np.float32),
                'feature_length': T,
                'raw_predict': greedy_decode_one(logits[local_i], T, BLANK_DECODE_PENALTY, UNK_DECODE_PENALTY),
            })
        if start == 0:
            print('First batch done. Example raw:', records[-len(batch_df)]['raw_predict'])
    return records

train_cache = run_manual_inference(train_df, batch_size=INFER_BATCH_SIZE)
valid_cache = run_manual_inference(valid_df, batch_size=INFER_BATCH_SIZE)
print('Cached train:', len(train_cache), 'valid:', len(valid_cache))

valid_ground_truth = valid_df[['canonical_norm', 'transcript_norm']].rename(
    columns={'canonical_norm': 'canonical', 'transcript_norm': 'transcript'}
).reset_index(drop=True)


## 8. CTC Viterbi Alignment


In [ ]:
def log_softmax_np(x):
    x = x.astype(np.float32)
    m = np.max(x, axis=-1, keepdims=True)
    y = x - m
    return y - np.log(np.sum(np.exp(y), axis=-1, keepdims=True))

def ctc_viterbi_align(logits, target_ids, blank_id=0):
    target_ids = [int(x) for x in target_ids if int(x) != blank_id]
    T = int(logits.shape[0])
    U = len(target_ids)
    if U == 0 or T == 0:
        return None
    ext = [blank_id]
    for y in target_ids:
        ext.append(y)
        ext.append(blank_id)
    S = len(ext)
    log_probs = log_softmax_np(logits)
    neg_inf = -1e30
    dp = np.full((T, S), neg_inf, dtype=np.float32)
    bp = np.full((T, S), -1, dtype=np.int32)
    dp[0, 0] = log_probs[0, blank_id]
    if S > 1:
        dp[0, 1] = log_probs[0, ext[1]]
        bp[0, 1] = 0
    for t in range(1, T):
        for s in range(S):
            best_prev = s
            best_score = dp[t - 1, s]
            if s - 1 >= 0 and dp[t - 1, s - 1] > best_score:
                best_score = dp[t - 1, s - 1]
                best_prev = s - 1
            if s - 2 >= 0 and ext[s] != blank_id and ext[s] != ext[s - 2] and dp[t - 1, s - 2] > best_score:
                best_score = dp[t - 1, s - 2]
                best_prev = s - 2
            dp[t, s] = best_score + log_probs[t, ext[s]]
            bp[t, s] = best_prev
    end_candidates = [S - 1, S - 2]
    end_s = max(end_candidates, key=lambda s: dp[T - 1, s])
    if dp[T - 1, end_s] <= neg_inf / 2:
        return None
    path = np.zeros(T, dtype=np.int32)
    s = end_s
    for t in range(T - 1, -1, -1):
        path[t] = s
        s = bp[t, s] if t > 0 else s
        if s < 0:
            s = 0
    segments = []
    for i in range(U):
        ext_pos = 2 * i + 1
        frames = np.where(path == ext_pos)[0]
        if len(frames) == 0:
            segments.append(None)
        else:
            segments.append((int(frames[0]), int(frames[-1]) + 1))
    return segments

def segment_embedding(hidden, segment, strategy=POOL_FRAME_STRATEGY):
    if segment is None:
        return None
    s, e = segment
    if e <= s:
        return None
    if strategy == 'mean':
        return hidden[s:e].mean(axis=0)
    mid = (s + e - 1) // 2
    return hidden[mid]

def safe_encode_tokens(tokens):
    return [phone2id.get(tok, phone2id[unk_token]) for tok in tokens]


## 9. Build Reference Pool From Correct Training Phones


In [ ]:
def build_reference_pool(train_records):
    by_phone = defaultdict(list)
    skipped_align = 0
    skipped_segment = 0
    for rec in train_records:
        transcript_tokens = eval_phone_tokens(rec['transcript_norm'])
        target_ids = safe_encode_tokens(transcript_tokens)
        correct_mask = transcript_correct_mask(rec['canonical_norm'], rec['transcript_norm'])
        if len(correct_mask) != len(transcript_tokens):
            skipped_align += 1
            continue
        segments = ctc_viterbi_align(rec['logits'], target_ids, phone2id[blank_token])
        if segments is None or len(segments) != len(transcript_tokens):
            skipped_align += 1
            continue
        for tok, tok_id, is_correct, seg in zip(transcript_tokens, target_ids, correct_mask, segments):
            if not is_correct or tok_id in (phone2id[blank_token], phone2id[unk_token]):
                continue
            emb = segment_embedding(rec['hidden'], seg)
            if emb is None:
                skipped_segment += 1
                continue
            by_phone[tok_id].append(emb.astype(np.float32))
    rng = np.random.default_rng(BASELINE_SEED)
    pool_embs = []
    pool_labels = []
    pool_counts = {}
    for tok_id, embs in by_phone.items():
        if len(embs) > POOL_MAX_PER_PHONE:
            idx = rng.choice(len(embs), size=POOL_MAX_PER_PHONE, replace=False)
            embs = [embs[i] for i in idx]
        pool_counts[id2phone[tok_id]] = len(embs)
        pool_embs.extend(embs)
        pool_labels.extend([tok_id] * len(embs))
    pool_embs = np.stack(pool_embs).astype(np.float32)
    norms = np.linalg.norm(pool_embs, axis=1, keepdims=True) + 1e-8
    pool_embs = pool_embs / norms
    pool_labels = np.array(pool_labels, dtype=np.int64)
    return pool_embs, pool_labels, pool_counts, {'skipped_align': skipped_align, 'skipped_segment': skipped_segment}

pool_embs_np, pool_labels_np, pool_counts, pool_stats = build_reference_pool(train_cache)
pool_embs = torch.tensor(pool_embs_np, dtype=torch.float32, device=DEVICE)
pool_labels = torch.tensor(pool_labels_np, dtype=torch.long, device=DEVICE)

print('Pool embeddings:', pool_embs.shape)
print('Unique pool phones:', len(pool_counts))
print('Pool stats:', pool_stats)
display(pd.DataFrame(Counter(pool_counts).most_common(20), columns=['phone', 'count']))


## 10. PER-MDD Style Correction


In [ ]:
def retrieve_neighbors(query_emb, k):
    q = torch.tensor(query_emb, dtype=torch.float32, device=DEVICE)
    q = q / (torch.linalg.norm(q) + 1e-8)
    sims = torch.matmul(pool_embs, q)
    k_eff = min(int(k), sims.numel())
    vals, idx = torch.topk(sims, k=k_eff)
    return vals.detach().cpu().numpy(), pool_labels[idx].detach().cpu().numpy()

def vote_retrieval(query_emb, canonical_id, k, similarity_threshold, majority_ratio, margin):
    sims, labels = retrieve_neighbors(query_emb, k)
    keep = sims >= similarity_threshold
    if not np.any(keep):
        return {
            'decision_id': canonical_id,
            'decision': 'fallback_no_neighbor',
            'top_phone': id2phone[canonical_id],
            'top_similarity': float(sims[0]) if len(sims) else 0.0,
            'majority_ratio': 0.0,
            'canonical_best_similarity': 0.0,
        }
    filt_sims = sims[keep]
    filt_labels = labels[keep]
    label_counts = Counter(filt_labels.tolist())
    top_label, top_count = label_counts.most_common(1)[0]
    ratio = top_count / len(filt_labels)
    top_sim = float(np.max(filt_sims[filt_labels == top_label]))
    canonical_sims = filt_sims[filt_labels == canonical_id]
    canonical_best = float(np.max(canonical_sims)) if len(canonical_sims) else -1.0
    if top_label != canonical_id and ratio >= majority_ratio and top_sim >= similarity_threshold and (top_sim - canonical_best) >= margin:
        return {
            'decision_id': int(top_label),
            'decision': 'retrieval_change',
            'top_phone': id2phone[int(top_label)],
            'top_similarity': top_sim,
            'majority_ratio': float(ratio),
            'canonical_best_similarity': canonical_best,
        }
    return {
        'decision_id': canonical_id,
        'decision': 'fallback_canonical',
        'top_phone': id2phone[int(top_label)],
        'top_similarity': top_sim,
        'majority_ratio': float(ratio),
        'canonical_best_similarity': canonical_best,
    }

def correct_record(rec, k=5, similarity_threshold=0.75, majority_ratio=0.8, margin=0.06, require_raw_agreement=True):
    canonical_tokens = eval_phone_tokens(rec['canonical_norm'])
    canonical_ids = safe_encode_tokens(canonical_tokens)
    raw_map = canonical_to_raw_map(rec['canonical_norm'], rec['raw_predict'])
    segments = ctc_viterbi_align(rec['logits'], canonical_ids, phone2id[blank_token])
    if segments is None or len(segments) != len(canonical_tokens):
        return rec['canonical_norm'], [{'decision': 'fallback_alignment_failed'}]
    corrected_ids = []
    debug_rows = []
    for pos, (tok, tok_id, seg) in enumerate(zip(canonical_tokens, canonical_ids, segments)):
        emb = segment_embedding(rec['hidden'], seg)
        raw_phone = raw_map[pos] if pos < len(raw_map) else None
        if emb is None or tok_id == phone2id[unk_token]:
            corrected_ids.append(tok_id)
            debug_rows.append({'pos': pos, 'canonical_phone': tok, 'raw_phone': raw_phone, 'decision': 'fallback_no_embedding'})
            continue
        vote = vote_retrieval(emb, tok_id, k, similarity_threshold, majority_ratio, margin)
        decision_id = vote['decision_id']
        if require_raw_agreement and decision_id != tok_id:
            raw_id = phone2id.get(raw_phone, None) if raw_phone is not None else None
            if raw_id != decision_id:
                vote['decision'] = 'fallback_raw_disagrees'
                decision_id = tok_id
        corrected_ids.append(decision_id)
        debug_rows.append({
            'pos': pos,
            'canonical_phone': tok,
            'raw_phone': raw_phone,
            'decision_phone': id2phone[int(decision_id)],
            **vote,
        })
    return ' '.join(id2phone[int(i)] for i in corrected_ids if int(i) != phone2id[blank_token]), debug_rows


## 11. Grid Search Retrieval Thresholds On Validation


In [ ]:
grid = []
for k in [3, 5, 10]:
    for similarity_threshold in [0.70, 0.75, 0.80, 0.85]:
        for majority_ratio in [0.6, 0.8]:
            for margin in [0.03, 0.06, 0.10]:
                for require_raw_agreement in [True, False]:
                    grid.append({
                        'k': k,
                        'similarity_threshold': similarity_threshold,
                        'majority_ratio': majority_ratio,
                        'margin': margin,
                        'require_raw_agreement': require_raw_agreement,
                    })

rows = []
best = None
best_predictions = None
best_debug = None

for cfg_i, cfg in enumerate(grid):
    preds = []
    debug_all = []
    for rec in valid_cache:
        pred, dbg = correct_record(rec, **cfg)
        preds.append(pred)
        for d in dbg:
            debug_all.append({'id': rec['id'], **d})
    results = pd.DataFrame({'predict': preds})
    metrics = compute_metrics_from_frames(valid_ground_truth, results)
    row = {**cfg, **metrics}
    rows.append(row)
    if best is None or metrics['Score'] > best['Score']:
        best = row
        best_predictions = preds
        best_debug = debug_all
    if (cfg_i + 1) % 20 == 0:
        print(f'Grid {cfg_i + 1}/{len(grid)} best Score={best["Score"]:.6f} F1={best["F1"]:.6f} Precision={best["Precision"]:.6f}')

metrics_df = pd.DataFrame(rows).sort_values('Score', ascending=False).reset_index(drop=True)
metrics_path = PER_MDD_DIR / 'per_mdd_grid_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

best_config = {
    'k': int(best['k']),
    'similarity_threshold': float(best['similarity_threshold']),
    'majority_ratio': float(best['majority_ratio']),
    'margin': float(best['margin']),
    'require_raw_agreement': bool(best['require_raw_agreement']),
}
best_config_path = PER_MDD_DIR / 'best_per_mdd_config.json'
best_config_path.write_text(json.dumps(best_config, indent=2), encoding='utf-8')

best_metrics_path = PER_MDD_DIR / 'best_per_mdd_metrics.json'
best_metrics_path.write_text(json.dumps(best, indent=2), encoding='utf-8')

corrected_valid_results = pd.DataFrame({'predict': best_predictions})
corrected_valid_results_path = PER_MDD_DIR / 'corrected_valid_results.csv'
corrected_valid_results.to_csv(corrected_valid_results_path, index=False)

best_debug_df = pd.DataFrame(best_debug)
best_debug_path = PER_MDD_DIR / 'per_mdd_valid_debug.csv'
best_debug_df.to_csv(best_debug_path, index=False)

print('Wrote:', metrics_path)
print('Wrote:', best_config_path)
print('Wrote:', best_metrics_path)
print('Wrote:', corrected_valid_results_path)
print('Wrote:', best_debug_path)
display(metrics_df.head(20))
print_metrics('Best corrected PER-MDD', best)


## 12. Load Public Test Metadata


In [ ]:
def find_public_test_root():
    candidates = [
        Path('/kaggle/input/mdd-challenge-2025-public-test/MDD-Challenge-2025-public-test'),
        Path('/kaggle/input/mdd-challenge-2025-public-test'),
        Path('/kaggle/input/mdd-challenge-data/MDD-Challenge-2025-public-test/MDD-Challenge-2025-public-test'),
        Path('/kaggle/input/mdd-challenge-data/MDD-Challenge-2025-public-test'),
        Path('./MDD-Challenge-2025-public-test/MDD-Challenge-2025-public-test'),
        Path('./MDD-Challenge-2025-public-test'),
        Path('.'),
    ]
    for p in candidates:
        if (p / 'metadata' / 'public_test_phones.csv').exists() and (p / 'audio_data' / 'public_test').exists():
            return p
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for phones_csv in kaggle_input.rglob('metadata/public_test_phones.csv'):
            root = phones_csv.parent.parent
            if (root / 'audio_data' / 'public_test').exists():
                return root
    for phones_csv in Path('.').rglob('metadata/public_test_phones.csv'):
        root = phones_csv.parent.parent
        if (root / 'audio_data' / 'public_test').exists():
            return root
    raise FileNotFoundError('Public test root not found. Please set PUBLIC_TEST_ROOT manually.')

PUBLIC_TEST_ROOT = find_public_test_root()
PUBLIC_TEST_META = PUBLIC_TEST_ROOT / 'metadata' / 'public_test_phones.csv'
PUBLIC_TEST_AUDIO_DIR = PUBLIC_TEST_ROOT / 'audio_data' / 'public_test'

public_test_phones = pd.read_csv(PUBLIC_TEST_META)
public_test_phones['canonical_norm'] = public_test_phones['canonical'].map(normalize_phone_text)
public_test_phones['file_name'] = public_test_phones['path'].map(lambda x: Path(str(x)).name)

file_name_counts = public_test_phones['file_name'].value_counts()
dup_files = file_name_counts[file_name_counts > 1]
if len(dup_files):
    raise ValueError(f'Duplicate file names in public_test_phones.csv: {dup_files.head(10).to_dict()}')

meta_by_file = public_test_phones.set_index('file_name', drop=False)

# Deterministic folder order: lexicographic file-name order under audio_data/public_test.
ordered_audio_files = sorted(PUBLIC_TEST_AUDIO_DIR.glob('*.wav'), key=lambda p: p.name)

rows = []
missing_meta = []
for output_id, audio_path in enumerate(ordered_audio_files):
    file_name = audio_path.name
    if file_name not in meta_by_file.index:
        missing_meta.append(file_name)
        continue
    meta_row = meta_by_file.loc[file_name]
    rows.append({
        'id': int(output_id),
        'source_id': meta_row['id'],
        'path': meta_row['path'],
        'audio_path': audio_path,
        'canonical_norm': meta_row['canonical_norm'],
        # Keep test transcript unavailable in the inference pipeline to avoid leakage.
        'transcript_norm': None,
    })

if missing_meta:
    raise ValueError(f'Missing metadata for {len(missing_meta)} public-test wav files. First few: {missing_meta[:10]}')

used_file_names = {Path(str(r['path'])).name for r in rows}
unused_meta = sorted(set(meta_by_file.index) - used_file_names)
if unused_meta:
    print('Unused public-test metadata rows:', len(unused_meta), unused_meta[:10])

test_df = pd.DataFrame(rows)
print('PUBLIC_TEST_ROOT:', PUBLIC_TEST_ROOT)
print('Ordered public-test rows:', len(test_df))
print('Public-test audio dir:', PUBLIC_TEST_AUDIO_DIR)
display(test_df[['id', 'source_id', 'path', 'canonical_norm']].head(20))


## 13. Predict Public Test


In [ ]:
test_cache = run_manual_inference(test_df, batch_size=INFER_BATCH_SIZE)
print('Cached test:', len(test_cache))

best_cfg = {
    'k': int(best['k']),
    'similarity_threshold': float(best['similarity_threshold']),
    'majority_ratio': float(best['majority_ratio']),
    'margin': float(best['margin']),
    'require_raw_agreement': bool(best['require_raw_agreement']),
}
print('Using best config:', best_cfg)

test_predictions = []
test_debug_rows = []
for rec, (_, row) in zip(test_cache, test_df.iterrows()):
    pred, dbg = correct_record(rec, **best_cfg)
    test_predictions.append(pred)
    for d in dbg:
        test_debug_rows.append({
            'id': rec['id'],
            'source_id': row['source_id'],
            'path': row['path'],
            **d,
        })

predictions_df = pd.DataFrame({
    'id': test_df['id'].astype(int),
    'predict': test_predictions,
})
predictions_path = PER_MDD_DIR / 'predictions.csv'
predictions_df.to_csv(predictions_path, index=False)

order_debug_df = test_df[['id', 'source_id', 'path']].copy()
order_debug_df['predict'] = test_predictions
order_debug_path = PER_MDD_DIR / 'public_test_order_debug.csv'
order_debug_df.to_csv(order_debug_path, index=False)

test_debug_df = pd.DataFrame(test_debug_rows)
test_debug_path = PER_MDD_DIR / 'public_test_per_mdd_debug.csv'
test_debug_df.to_csv(test_debug_path, index=False)

print('Wrote:', predictions_path)
print('Wrote:', order_debug_path)
print('Wrote:', test_debug_path)
display(predictions_df.head(20))
